# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipos — Semanas RAG**

---

* **Nombres y matrículas:**

  *   Jose Angel Barajas A01797221
  *   Elemento de lista
  *   Elemento de lista

* **Número de Equipo:**

---

## 📋 v6_grok — RAG Anti-Alucinación con Grok API + Extracción de Tablas

### Historial de versiones

| Versión | LLM | Cambio principal |
|---------|-----|-----------------|
| v1–v3 | `qwen2.5-coder-7b` local | Técnicas anti-hallucination (fallaron con modelo pequeño) |
| v4 | `qwen2.5-coder-7b` local | RetrievalQA básico (funcional, sin anti-hallucination) |
| v5_grok | `grok-4-1-fast-non-reasoning` | Grok API + anti-hallucination (técnicas validadas) |
| **v6_grok** | `grok-4-1-fast-non-reasoning` | **+Tablas PDF + MMR + Caché + Grounding Check** |

### Mejoras de v6_grok respecto a v5_grok

| # | Mejora | Descripción |
|---|--------|-------------|
| 1 | **Extracción de tablas** | `pdfplumber` extrae tablas como Markdown; PyPDFLoader para texto |
| 2 | **Text splitter inteligente** | Tablas sin dividir; texto con chunk_size=1200/overlap=200 |
| 3 | **Caché del VectorDB** | Evita reconstruir el índice en cada pregunta |
| 4 | **Cosine similarity** | `normalize_embeddings=True` + espacio coseno en ChromaDB |
| 5 | **MMR retrieval** | Maximiza relevancia Y diversidad de chunks |
| 6 | **Umbral dinámico** | Percentil P40 en lugar de threshold fijo 0.0 |
| 7 | **Grounding Check** | Verifica que la respuesta esté anclada al contexto |
| 8 | **Prompt v6 mejorado** | Maneja info parcial; sección PROHIBIDO explícita |
| 9 | **Temperatura 0.1** | Reducida desde 0.3 en v5 para mayor fidelidad |
| 10 | **MAX_CHARS=14000** | Aumentado desde 12000 para contexto más rico |

> **Hipótesis de v6_grok:** Extracción de tablas + MMR + Grounding Check mejoran la calidad y
> verificabilidad de las respuestas, manteniendo Grok como único proveedor LLM.

---

---

## 🧩 Step 1 – Instalación de Paquetes

**Paquete nuevo en v6_grok:**
- `pdfplumber` → Extracción de tablas en formato Markdown estructurado

In [ ]:
import sys

!{sys.executable} -m pip install langchain langchain-community langchain-text-splitters langchain-huggingface langchain-openai langchain-core -q
!{sys.executable} -m pip install chromadb pypdf sentence-transformers gradio openai python-dotenv pdfplumber -q

---

## ⚙️ Step 2 – Imports y Configuración

Carga de librerías y configuración de la API key de xAI desde el archivo `.env`.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate

import requests
import os
import re
import warnings
import pdfplumber
import numpy as np
from dotenv import load_dotenv
import pathlib
import gradio as gr

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"

load_dotenv()
GROK_KEY   = os.getenv("xAI_API_KEY")
GROK_URL   = "https://api.x.ai/v1/chat/completions"
GROK_MODEL = "grok-4-1-fast-non-reasoning"

print(f"✅ Imports completados")
print(f"🔑 xAI_API_KEY: {'cargada ✅' if GROK_KEY else '❌ NO encontrada — agrega xAI_API_KEY al .env'}")
print(f"🤖 Modelo: {GROK_MODEL}")

---

## 🧠 Step 3 – Configurar Grok LLM via xAI API

**Modelo:** `grok-4-1-fast-non-reasoning` (xAI API, endpoint compatible con OpenAI)
**Temperatura:** 0.1 — máxima fidelidad al contexto (reducida desde 0.3 en v5)

In [ ]:
def grok_chat(messages, temperature=0.1, max_tokens=2048):
    """Llamada directa a la API de xAI Grok (endpoint compatible con OpenAI)."""
    if not GROK_KEY:
        raise ValueError(
            "xAI_API_KEY no configurada.\n"
            "Agrega la clave al archivo .env: xAI_API_KEY=xai-..."
        )
    headers = {
        "Authorization": f"Bearer {GROK_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model":       GROK_MODEL,
        "messages":    messages,
        "temperature": temperature,
        "max_tokens":  max_tokens,
    }
    response = requests.post(GROK_URL, headers=headers, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


# ─── Test de conexión ────────────────────────────────────────────────────────
print(f"Probando conexión con Grok ({GROK_MODEL})...")
try:
    _test = grok_chat([{"role": "user", "content": "Dame una respuesta corta en español confirmando que la API de xAI funciona."}])
    print(f"✅ Grok response: {_test[:120]}")
except Exception as e:
    print(f"❌ Error: {e}")

---

## 🛡️ Step 4 – Prompt Anti-Alucinación v6 (MEJORADO)

### Cambios respecto a v5_grok

| Aspecto | v5_grok | v6_grok |
|---------|---------|---------|
| Manejo de info parcial | No contemplado | Instrucción explícita para info parcial |
| Reglas prohibidas | Implícitas en el tono | Sección "PROHIBIDO ABSOLUTAMENTE" explícita |
| Requisito de citas | Sugerido | Obligatorio: sin citar [doc:N] → no válido |
| Foco del asistente | General | Especializado en Python y ML |

In [ ]:
ANTI_HALLUCINATION_PROMPT_V6 = """Eres un asistente especializado en Python y Machine Learning que responde EXCLUSIVAMENTE con información de los documentos de referencia proporcionados.

CONTEXTO DISPONIBLE:
{context}

PREGUNTA: {question}

INSTRUCCIONES DE RESPUESTA (sigue estas reglas en orden):

1. LEE el contexto cuidadosamente antes de responder.

2. Si encuentras información DIRECTA y COMPLETA en el contexto:
   - Responde de forma clara y estructurada
   - Cita la fuente entre corchetes: [doc:0], [doc:1], etc., después de cada dato específico
   - Usa el mismo idioma de la pregunta

3. Si encuentras información PARCIAL (el contexto toca el tema pero no responde completamente):
   - Proporciona lo que SÍ encontraste, citando fuentes
   - Indica explícitamente: "Nota: El contexto contiene información parcial sobre este tema."

4. Si el contexto NO contiene información relevante:
   - Responde exactamente: "Los documentos de referencia proporcionados no contienen información suficiente para responder esta pregunta."
   - NO uses tu conocimiento propio para completar la respuesta

5. PROHIBIDO ABSOLUTAMENTE:
   - Inventar ejemplos de código que no estén en el contexto
   - Mencionar librerías, funciones o conceptos no presentes en los documentos
   - Añadir información "de relleno" con tu conocimiento previo
   - Responder sin citar al menos una fuente [doc:N]

RESPUESTA:"""

PROMPT_TEMPLATE_V6 = PromptTemplate(
    input_variables=["context", "question"],
    template=ANTI_HALLUCINATION_PROMPT_V6
)

print("✅ Prompt anti-alucinación v6 configurado.")
print(f"📐 Longitud del prompt: {len(ANTI_HALLUCINATION_PROMPT_V6)} caracteres")

---

## 📄 Step 5 – Carga Híbrida de Documentos PDF con Extracción de Tablas (NUEVO v6)

### ¿Por qué PyPDFLoader solo no es suficiente?

Los cheat sheets contienen **tablas críticas** que PyPDFLoader extrae como texto plano,
perdiendo la estructura columnar. `pdfplumber` detecta y preserva tablas en Markdown.

| Extractor | Texto corrido | Tablas |
|-----------|---------------|--------|
| PyPDFLoader | ✅ Excelente | ❌ Pierde estructura |
| pdfplumber | ❌ No se usa para texto | ✅ Markdown estructurado |
| **v6_grok (híbrido)** | ✅ PyPDFLoader | ✅ pdfplumber |

In [ ]:
def extract_tables_pdfplumber(file_path: str) -> list:
    """Extrae tablas de un PDF con pdfplumber y las convierte a Markdown."""
    table_docs  = []
    source_name = os.path.basename(file_path)
    with pdfplumber.open(file_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            for table_idx, table in enumerate(page.extract_tables()):
                if not table or len(table) < 2:
                    continue
                header   = [str(c).strip() if c else "" for c in table[0]]
                md_lines = [
                    "| " + " | ".join(header) + " |",
                    "| " + " | ".join(["---"] * len(header)) + " |",
                ]
                for row in table[1:]:
                    row = [str(c).strip().replace("\n", " ") if c else "" for c in row]
                    md_lines.append("| " + " | ".join(row) + " |")
                table_docs.append(Document(
                    page_content=f"[TABLA] Página {page_num + 1}, Tabla {table_idx + 1}:\n" + "\n".join(md_lines),
                    metadata={
                        "source_file":  source_name,
                        "page":         page_num,
                        "table_index":  table_idx,
                        "is_table":     True,
                        "content_type": "table",
                    }
                ))
    return table_docs


def document_loader_v6(file_path: str) -> list:
    """Pipeline híbrido: PyPDFLoader (texto) + pdfplumber (tablas)."""
    source_name = os.path.basename(file_path)
    print(f"\n📂 Cargando: {source_name}")

    # 1. Texto general con PyPDFLoader
    loader    = PyPDFLoader(file_path)
    text_docs = loader.load()
    for doc in text_docs:
        doc.metadata.update({"source_file": source_name, "is_table": False, "content_type": "text"})
    print(f"   📃 Páginas de texto: {len(text_docs)}")

    # 2. Tablas con pdfplumber
    try:
        table_docs = extract_tables_pdfplumber(file_path)
        print(f"   📊 Tablas extraídas: {len(table_docs)}")
        if table_docs:
            print(f"   🔍 Preview primera tabla:\n{table_docs[0].page_content[:300]}")
    except Exception as e:
        print(f"   ⚠️  Error extrayendo tablas: {e}")
        table_docs = []

    all_docs = text_docs + table_docs
    print(f"   ✅ Total: {len(all_docs)} documentos ({len(text_docs)} texto + {len(table_docs)} tablas)")
    return all_docs

---

## ✂️ Step 6 – Text Splitter con Tratamiento Especial para Tablas (NUEVO v6)

| Parámetro | v5_grok | v6_grok | Justificación |
|-----------|---------|---------|---------------|
| `chunk_size` | 1000 | 1200 | Mayor contexto por chunk |
| `chunk_overlap` | 150 | 200 | Mejor continuidad semántica |
| Tablas | Se dividen | **No se dividen** | Preserva estructura tabular completa |

In [ ]:
def text_splitter_v6(docs: list) -> list:
    """Divide docs en chunks; preserva tablas intactas."""
    text_docs  = [d for d in docs if not d.metadata.get("is_table", False)]
    table_docs = [d for d in docs if d.metadata.get("is_table", False)]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1200,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    text_chunks  = splitter.split_documents(text_docs)
    table_chunks = table_docs  # Sin dividir

    all_chunks  = text_chunks + table_chunks
    text_sizes  = [len(c.page_content) for c in text_chunks]
    table_sizes = [len(c.page_content) for c in table_chunks]

    print(f"✅ Chunks generados:")
    print(f"   📃 Texto:  {len(text_chunks)} chunks — promedio {int(np.mean(text_sizes)) if text_sizes else 0} chars")
    print(f"   📊 Tablas: {len(table_chunks)} chunks (sin dividir) — promedio {int(np.mean(table_sizes)) if table_sizes else 0} chars")
    print(f"   🔢 TOTAL:  {len(all_chunks)} chunks")
    return all_chunks

---

## 🧠 Step 7 – Embeddings + VectorDB con Caché y Cosine Similarity (MEJORADO v6)

| Aspecto | v5_grok | v6_grok |
|---------|---------|---------|
| Normalización de embeddings | ❌ No | ✅ `normalize_embeddings=True` |
| Métrica de similitud | L2 (por defecto) | ✅ Cosine |
| Caché del VectorDB | ❌ Reconstruye siempre | ✅ Caché en memoria por sesión |

In [ ]:
_vectordb_cache = {}


def get_embedding_model():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )


def build_vectordb(file_paths: list, force_rebuild: bool = False) -> Chroma:
    """Construye o recupera del caché el vector store ChromaDB."""
    global _vectordb_cache
    cache_key = tuple(sorted(file_paths))

    if cache_key in _vectordb_cache and not force_rebuild:
        print("⚡ Usando vector store en caché (evita re-procesar PDFs)")
        return _vectordb_cache[cache_key]

    print("🔨 Construyendo vector store desde cero...")
    all_docs = []
    for fp in file_paths:
        all_docs.extend(document_loader_v6(fp))

    chunks = text_splitter_v6(all_docs)

    print("\n⏳ Generando embeddings (puede tardar 1-2 minutos la primera vez)...")
    embed    = get_embedding_model()
    vectordb = Chroma.from_documents(
        documents=chunks,
        embedding=embed,
        collection_metadata={"hnsw:space": "cosine"},
    )
    print(f"✅ Vector store construido con {vectordb._collection.count()} vectores")
    _vectordb_cache[cache_key] = vectordb
    return vectordb

---

## 🔍 Step 8 – Retriever con MMR, Umbral Dinámico y Grounding Check (NUEVO v6)

### Mejoras clave respecto a v5_grok

| Técnica | v5_grok | v6_grok |
|---------|---------|---------|
| **Algoritmo de búsqueda** | `similarity_search` | ✅ MMR (`max_marginal_relevance_search`) |
| **Threshold** | Fijo (0.0) | ✅ Dinámico (percentil P40) |
| **Verificación post-generación** | ❌ No | ✅ Grounding Check |

**MMR (Maximal Marginal Relevance):** Selecciona chunks relevantes para la pregunta Y
diferentes entre sí, evitando redundancia.

**Grounding Check:** Verifica qué fracción de los términos clave de la respuesta aparecen
en el contexto recuperado. Score < 0.35 → advertencia de posible alucinación.

In [ ]:
MAX_CHUNKS          = 8
MAX_CHARS           = 14000
MMR_LAMBDA          = 0.6    # 0 = máxima diversidad, 1 = máxima relevancia
GROUNDING_THRESHOLD = 0.35


def retrieve_with_mmr(vectordb: Chroma, question: str, k: int = MAX_CHUNKS) -> tuple:
    """Recupera chunks con MMR + umbral dinámico. Retorna (chunks, scores)."""
    # Scores para diagnóstico y umbral dinámico
    try:
        scored = vectordb.similarity_search_with_relevance_scores(question, k=k * 3)
        scores = [round(s, 4) for _, s in scored]
        print(f"\n📊 Scores de similitud (top {len(scores)}): {scores}")
        dynamic_threshold = float(np.percentile(scores, 40)) if scores else 0.05
        dynamic_threshold = max(dynamic_threshold, 0.05)
        print(f"   🎯 Umbral dinámico (P40): {dynamic_threshold:.4f}")
    except Exception as e:
        print(f"   ⚠️  Error calculando scores: {e}")
        scores = []

    # MMR con fallback a similarity_search
    try:
        chunks = vectordb.max_marginal_relevance_search(
            question, k=k, fetch_k=k * 3, lambda_mult=MMR_LAMBDA
        )
        print(f"   ✅ MMR recuperó {len(chunks)} chunks")
    except Exception as e:
        print(f"   ⚠️  MMR falló ({e}), usando similarity_search de respaldo")
        chunks = vectordb.similarity_search(question, k=k)

    # Trim a MAX_CHARS
    trimmed, total_chars = [], 0
    for chunk in chunks:
        if total_chars + len(chunk.page_content) > MAX_CHARS:
            break
        trimmed.append(chunk)
        total_chars += len(chunk.page_content)

    n_tables = sum(1 for c in trimmed if c.metadata.get("is_table", False))
    n_text   = len(trimmed) - n_tables
    print(f"   📤 Chunks al LLM: {len(trimmed)} ({total_chars} chars) — 📃 texto: {n_text} | 📊 tablas: {n_tables}")
    return trimmed, scores


def grounding_check(answer: str, context_docs: list) -> dict:
    """Verifica que los términos clave de la respuesta aparezcan en el contexto."""
    STOPWORDS = {
        "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
        "have", "has", "had", "do", "does", "did", "will", "would", "could",
        "should", "may", "might", "shall", "can", "need", "and", "or", "but",
        "if", "in", "on", "at", "to", "for", "of", "with", "by", "from", "as",
        "el", "la", "los", "las", "un", "una", "de", "del", "en", "con", "por",
        "para", "que", "se", "es", "son", "fue", "ser", "sus", "al", "lo",
        "this", "that", "these", "those", "it", "its", "they", "them", "their",
        "which", "who", "what", "when", "where", "how", "why", "not", "no",
        "also", "use", "used", "using", "i", "you", "we", "he", "she",
        "doc", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9",
        "more", "also", "note", "pero", "más", "ya", "si", "muy",
    }
    answer_words = re.findall(r'\b[a-zA-ZáéíóúÁÉÍÓÚñÑ]{4,}\b', answer.lower())
    key_terms    = [w for w in answer_words if w not in STOPWORDS]

    if not key_terms:
        return {"score": 1.0, "grounded": True, "note": "No hay términos clave que verificar",
                "grounded_terms": 0, "total_key_terms": 0, "ungrounded_sample": []}

    context_text   = " ".join(d.page_content.lower() for d in context_docs)
    unique_key     = list(set(key_terms))
    grounded       = [t for t in unique_key if t in context_text]
    ungrounded     = [t for t in unique_key if t not in context_text]
    score          = len(grounded) / len(unique_key)

    return {
        "score":            round(score, 3),
        "grounded":         score >= GROUNDING_THRESHOLD,
        "total_key_terms":  len(unique_key),
        "grounded_terms":   len(grounded),
        "ungrounded_sample": list(set(ungrounded))[:10],
        "note":             f"{len(grounded)}/{len(unique_key)} términos clave encontrados en el contexto",
    }


print("✅ Retriever MMR y Grounding Check configurados.")
print(f"   MAX_CHUNKS={MAX_CHUNKS} | MAX_CHARS={MAX_CHARS} | MMR_LAMBDA={MMR_LAMBDA} | GROUNDING_THRESHOLD={GROUNDING_THRESHOLD}")

---

## 🔗 Step 9 – QA Chain Anti-Alucinación v6_grok

### Flujo completo de una consulta

```
Pregunta del usuario
       │
       ▼
  [build_vectordb]  ──→  Caché o reconstrucción
       │
       ▼
  [retrieve_with_mmr]  ──→  MMR + umbral dinámico P40
       │
       ▼
  [ANTI_HALLUCINATION_PROMPT_V6]  ──→  Contexto con [doc:N] + marcadores 📊/📃
       │
       ▼
  [grok_chat (temp=0.1)]  ──→  Respuesta con citas
       │
       ▼
  [grounding_check]  ──→  Score de anclaje (🟢/🟡/🔴)
       │
       ▼
  Respuesta formateada + fuentes + score
```

In [ ]:
_conversation_history_v6 = []


def answer_question_v6(file_paths: list, question: str, verbose: bool = True) -> str:
    """Pipeline completo de QA anti-alucinación v6_grok."""
    global _conversation_history_v6

    # 1. VectorDB (con caché)
    vectordb = build_vectordb(file_paths)

    # 2. Retrieval MMR
    context_docs, scores = retrieve_with_mmr(vectordb, question)
    if not context_docs:
        return "No se recuperaron documentos. Verifica que los PDFs estén cargados correctamente."

    # 3. Contexto con markers [doc:N]
    context_parts = []
    for i, doc in enumerate(context_docs):
        source = doc.metadata.get("source_file", "unknown")
        page   = doc.metadata.get("page", "?")
        tag    = "📊 TABLA" if doc.metadata.get("is_table", False) else "📃 Texto"
        context_parts.append(
            f"[doc:{i}] ({tag} | Fuente: {source} | Pág. {page}):\n{doc.page_content}"
        )
    context = "\n\n---\n\n".join(context_parts)

    # 4. Construir mensajes con historial (últimos 3 turnos = 6 mensajes)
    full_prompt = ANTI_HALLUCINATION_PROMPT_V6.format(context=context, question=question)
    messages = []
    for role, content in _conversation_history_v6[-6:]:
        messages.append({"role": role, "content": content})
    messages.append({"role": "user", "content": full_prompt})

    # 5. Llamar a Grok
    try:
        answer = grok_chat(messages, temperature=0.1, max_tokens=2048)
    except requests.exceptions.HTTPError as e:
        code = e.response.status_code if e.response is not None else 0
        if code == 401:
            return "❌ Error 401: API Key inválida. Verifica xAI_API_KEY en .env"
        return f"❌ Error HTTP {code}: {e.response.text if e.response else str(e)}"
    except Exception as e:
        return f"❌ Error inesperado: {str(e)}"

    # 6. Grounding Check
    grounding = grounding_check(answer, context_docs)
    if verbose:
        print(f"\n🔍 Grounding Check:")
        print(f"   Score: {grounding['score']:.3f} | Anclado: {grounding['grounded']}")
        print(f"   {grounding['note']}")
        if not grounding['grounded'] and grounding.get('ungrounded_sample'):
            print(f"   ⚠️  Términos no encontrados en ctx: {grounding['ungrounded_sample'][:5]}")

    # 7. Historial conversacional (mantener últimos 6 turnos = 12 mensajes)
    _conversation_history_v6.append(("user", question))
    _conversation_history_v6.append(("assistant", answer))
    if len(_conversation_history_v6) > 12:
        _conversation_history_v6 = _conversation_history_v6[-12:]

    # 8. Formatear respuesta final
    sources       = sorted(set(d.metadata.get("source_file", "?") for d in context_docs))
    table_sources = list(set(
        d.metadata.get("source_file", "?")
        for d in context_docs if d.metadata.get("is_table", False)
    ))

    grounding_note = ""
    if not grounding['grounded']:
        grounding_note = (
            f"\n\n> ⚠️ **Advertencia de verificación:** Score de grounding "
            f"{grounding['score']:.2f} (umbral: {GROUNDING_THRESHOLD}). "
            f"Algunos términos podrían no estar directamente en los documentos proporcionados."
        )

    emoji = "🟢" if grounding['score'] >= 0.6 else ("🟡" if grounding['score'] >= GROUNDING_THRESHOLD else "🔴")

    final_answer = (
        f"{answer}"
        f"{grounding_note}"
        f"\n\n---"
        f"\n📎 **Fuentes consultadas:** {', '.join(sources)}"
        f"\n{emoji} **Grounding:** {grounding['score']:.2f} "
        f"({grounding['grounded_terms']}/{grounding['total_key_terms']} términos verificados)"
    )
    if table_sources:
        final_answer += f"\n📊 **Incluye tablas de:** {', '.join(sorted(table_sources))}"

    return final_answer


print("✅ Pipeline QA v6_grok configurado.")
print(f"   Temperatura: 0.1 | Max chunks: {MAX_CHUNKS} | Max chars: {MAX_CHARS}")

---

## 💻 Step 10 – Interfaz Gradio

**Mejoras respecto a v5_grok:**
- Ejemplos de preguntas precargados (6 preguntas de la actividad)
- Reset automático del historial al subir nuevos PDFs

In [ ]:
_gradio_files_v6 = None


def gradio_rag_v6(message: str, history: list, files) -> str:
    """
    Función principal de la interfaz Gradio.
    Firma requerida por gr.ChatInterface: (message, history, *additional_inputs)
    """
    global _gradio_files_v6, _conversation_history_v6

    if files is not None:
        new_files = files if isinstance(files, list) else [files]
        if new_files != _gradio_files_v6:
            _gradio_files_v6 = new_files
            _conversation_history_v6 = []  # Reset historial al cambiar PDFs
            print(f"📂 Nuevos PDFs cargados: {[os.path.basename(f) for f in new_files]}")

    if not _gradio_files_v6:
        return "⚠️ Por favor sube al menos un archivo PDF antes de hacer preguntas."

    try:
        return answer_question_v6(_gradio_files_v6, message, verbose=False)
    except Exception as e:
        err = str(e)
        if "Connection" in err or "10061" in err or "refused" in err:
            return "❌ No se puede conectar a xAI API. Verifica tu conexión y API key."
        return f"❌ Error: {err}" 

In [ ]:
gr.close_all()

rag_app_v6_grok = gr.ChatInterface(
    fn=gradio_rag_v6,
    additional_inputs=[
        gr.File(
            label="📂 Subir PDF(s) — python_cheatsheet.pdf y/o ml_cheatsheet.pdf",
            file_count="multiple",
            file_types=[".pdf"],
            type="filepath",
        ),
    ],
    title="🤖 ITESM-NLP RAG Chatbot v6_grok — Anti-Alucinación + Extracción de Tablas",
    description=(
        "**Chatbot RAG con Grok API para hojas de referencia de Python y Machine Learning**\n\n"
        "📋 **Instrucciones:**\n"
        "1. Sube `python_cheatsheet.pdf` y/o `ml_cheatsheet.pdf` usando el selector de archivos\n"
        "2. Escribe tu pregunta en el campo de texto\n"
        "3. La respuesta incluirá citas [doc:N] y un score de grounding\n\n"
        "🛡️ **Anti-alucinación v6:** Temp=0.1 | MMR Retrieval | Grounding Check | Tablas Markdown | Prompt estricto"
    ),
    examples=[
        ["According to the Exceptions section, what exception is raised when dividing by zero?"],
        ["Can you give me 2 examples of string methods from the Python cheat sheet?"],
        ["What are the main disadvantages of Random Forests?"],
        ["What are 3 use cases of clustering in Unsupervised Learning?"],
        ["What are the main data types in Python? Give examples of each."],
        ["What are the evaluation metrics for classification models and what does each measure?"],
    ],
)

rag_app_v6_grok.launch(server_name="127.0.0.1", server_port=7866, share=False)

⚡ Usando vector store en caché (evita re-procesar PDFs)

📊 Scores de similitud (top 24): [0.3188, 0.3188, 0.1669, 0.1669, 0.1455, 0.1455, 0.1243, 0.1243, 0.1074, 0.1074, 0.1054, 0.1054, 0.0789, 0.0789, 0.0755, 0.0755, 0.0686, 0.0686, 0.0649, 0.064, 0.064, 0.0567, 0.0567, 0.0497]
   🎯 Umbral dinámico (P40): 0.0762
   ✅ MMR recuperó 8 chunks
   📤 Chunks al LLM: 4 (10585 chars) — 📃 texto: 3 | 📊 tablas: 1
⚡ Usando vector store en caché (evita re-procesar PDFs)

📊 Scores de similitud (top 24): [0.319, 0.319, 0.3159, 0.3159, 0.3091, 0.3091, 0.2864, 0.2864, 0.2773, 0.2773, 0.2617, 0.2617, 0.2206, 0.2201, 0.2201, 0.2192, 0.2192, 0.2043, 0.2043, 0.1997, 0.1997, 0.1744, 0.1744, 0.1621]
   🎯 Umbral dinámico (P40): 0.2201
   ✅ MMR recuperó 8 chunks
   📤 Chunks al LLM: 8 (11509 chars) — 📃 texto: 6 | 📊 tablas: 2
⚡ Usando vector store en caché (evita re-procesar PDFs)

📊 Scores de similitud (top 24): [0.3304, 0.3304, 0.3204, 0.3204, 0.3142, 0.3142, 0.3001, 0.3001, 0.2918, 0.2918, 0.241, 0.241, 0.2269

---

### 🔴 Detener el servidor Gradio

Ejecuta la celda siguiente para liberar el puerto.

In [ ]:
gr.close_all()
try:
    rag_app_v6_grok.close()
    print("✅ Servidor Gradio cerrado correctamente.")
except Exception:
    print("✅ Servidor ya estaba cerrado.")

---

## 📝 Notas de v6_grok — Comparativa técnica con v5_grok

### ¿Qué cambió y por qué?

| Componente | v5_grok | v6_grok | Impacto esperado |
|-----------|---------|---------|-----------------|
| Carga de PDF | PyPDFLoader solo | Híbrido (PyPDFLoader + pdfplumber) | Captura información tabular |
| Text splitter | 1000/150, sin distinción | 1200/200, tablas sin dividir | Mejor preservación de estructura |
| Embeddings | Sin normalizar, L2 | Normalizados, cosine | Scores de similitud más confiables |
| VectorDB | Sin caché | Caché por sesión | Respuestas mucho más rápidas |
| Retrieval | Similarity + threshold 0.0 | MMR + umbral P40 dinámico | Mayor diversidad, menos redundancia |
| Verificación | No | Grounding Check (score + emoji) | Detecta posibles alucinaciones |
| Prompt | Suavizado (fix v5) | Estricto v6 con info parcial | Mejor control sin rechazar respondibles |
| Temperatura | 0.3 | 0.1 | Mayor fidelidad al contexto |
| LLM | Grok-4.1 Fast Non-Reasoning | Grok-4.1 Fast Non-Reasoning | Sin cambio (mismo modelo) |

*v6_grok — Grok Anti-Hallucination RAG + Extracción de Tablas — 2026-06-28*